#**Simple interpretable baseline**
In this approach, we model seniority and domain prediction as a text classification problem using job titles and descriptions. We apply Bag-of-Words and TF–IDF representations to transform raw text into numerical features, followed by standard classifiers such as Logistic Regression, Random Forest, and CatBoost. This approach serves as a strong and interpretable baseline, allowing us to evaluate how much signal is already contained in textual job information without heavy feature engineering.

### Preparing the dataset

In [22]:
!pip install catboost
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


In [23]:
with open("/content/linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)

jobs = []
for cv in cvs:
    for job in cv:
        jobs.append(job)

df = pd.DataFrame(jobs)
df_active = df[df["status"] == "ACTIVE"]
df_active.head()


,organization,linkedin,position,startDate,endDate,status,department,seniority
0,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokurist,2019-08,None,ACTIVE,Other,Management
1,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management
2,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Betriebswirtin,2019-07,None,ACTIVE,Other,Professional
3,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,Prokuristin,2019-07,None,ACTIVE,Other,Management
4,Depot4Design GmbH,https://www.linkedin.com/company/depot4design-...,CFO,2019-07,None,ACTIVE,Other,Management


##A. Senority Prediction

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

#Spliting Data
X_sn = df_active["position"]
y_sn = df_active["seniority"]

X_train_sn, X_test_sn, y_train_sn, y_test_sn = train_test_split(
    X_sn, y_sn, test_size=0.2, random_state=42, stratify=y_sn)

# Encode y
le = LabelEncoder()
y_train_enc_sn = le.fit_transform(y_train_sn)
y_test_enc_sn = le.transform(y_test_sn)

#define model
random_forest = RandomForestClassifier(n_estimators=200, random_state=42)
CatBoost = CatBoostClassifier(verbose=False,random_seed=42)
Logistic_Regression = LogisticRegression(max_iter=1000)

# Train pipeline Bag-of-Words
for model in [random_forest, CatBoost,Logistic_Regression]:
    pipeline = Pipeline([
        ("bow", CountVectorizer(lowercase=True, stop_words='english')),
        ("model", model)
    ])
    pipeline.fit(X_train_sn, y_train_enc_sn)
    y_pred_sn_bow = pipeline.predict(X_test_sn)
    acc = accuracy_score(y_test_enc_sn, y_pred_sn_bow)
    print(f"{type(model).__name__} Accuracy (BoW): {acc:.3f}")

# Train pipeline with TF–IDF
for model in [random_forest, CatBoost,Logistic_Regression]:
    pipeline = Pipeline([
        ("tfidf",TfidfVectorizer(lowercase=True, stop_words='english')),
        ("model", model)
    ])
    pipeline.fit(X_train_sn, y_train_enc_sn)
    y_pred_sn = pipeline.predict(X_test_sn)
    acc_tf = accuracy_score(y_test_enc_sn, y_pred_sn)
    print(f"{type(model).__name__} Accuracy (TF–IDF): {acc_tf:.3f}")

RandomForestClassifier Accuracy (BoW): 0.816
CatBoostClassifier Accuracy (BoW): 0.808
LogisticRegression Accuracy (BoW): 0.792
RandomForestClassifier Accuracy (TF–IDF): 0.808
CatBoostClassifier Accuracy (TF–IDF): 0.784
LogisticRegression Accuracy (TF–IDF): 0.760


## B. Domain Prediction

In [36]:
#Spliting Data
X_dm = df_active["position"]
y_dm = df_active["department"]

X_train_dm, X_test_dm, y_train_dm, y_test_dm = train_test_split(
    X_dm, y_dm, test_size=0.2, random_state=42, stratify=y_dm)

# Encode y
y_train_enc_dm = le.fit_transform(y_train_dm)
y_test_enc_dm = le.transform(y_test_dm)

#define model
random_forest = RandomForestClassifier(n_estimators=200, random_state=42)
CatBoost = CatBoostClassifier(verbose=False,random_seed=42)
Logistic_Regression = LogisticRegression(max_iter=6000)

# Train pipeline Bag-of-Words
for model in [random_forest, CatBoost,Logistic_Regression]:
    pipeline = Pipeline([
        ("bow", CountVectorizer(lowercase=True, stop_words='english')),
        ("model", model)
    ])
    pipeline.fit(X_train_dm, y_train_enc_dm)
    y_pred_dm_bow = pipeline.predict(X_test_dm)
    acc_dm_bow = accuracy_score(y_test_enc_dm, y_pred_dm_bow)
    print(f"{type(model).__name__} Accuracy (BoW): {acc_dm_bow:.3f}")

# Train pipeline with TF–IDF
for model in [random_forest, CatBoost,Logistic_Regression]:
    pipeline = Pipeline([
        ("tfidf",TfidfVectorizer(lowercase=True, stop_words='english')),
        ("model", model)
    ])
    pipeline.fit(X_train_dm, y_train_enc_dm)
    y_pred_dm_tf = pipeline.predict(X_test_dm)
    acc_dm_tf = accuracy_score(y_test_enc_dm, y_pred_dm_tf)
    print(f"{type(model).__name__} Accuracy (TF–IDF): {acc_dm_tf:.3f}")

RandomForestClassifier Accuracy (BoW): 0.768
CatBoostClassifier Accuracy (BoW): 0.736
LogisticRegression Accuracy (BoW): 0.720
RandomForestClassifier Accuracy (TF–IDF): 0.776
CatBoostClassifier Accuracy (TF–IDF): 0.720
LogisticRegression Accuracy (TF–IDF): 0.632


#